# 03 Training IndoBERT

Notebook ini digunakan untuk fine-tuning model IndoBERT pada data ulasan
Bank BJB yang sudah dibersihkan dan memiliki label sentimen.


## Catatan

- Pastikan dependency di `requirements.txt` sudah terpasang.
- Pastikan file `data/processed/bjb_reviews_clean.csv` sudah berisi kolom `content_clean` dan `label`.
- Notebook ini memakai skema default klasifikasi 3 kelas: `negative`, `neutral`, `positive`.


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_PATH = PROJECT_ROOT / "data" / "processed" / "bjb_reviews_clean.csv"
MODEL_DIR = PROJECT_ROOT / "models" / "indobert"
MODEL_NAME = "indobenchmark/indobert-base-p1"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

DATA_PATH, MODEL_DIR


In [ ]:
import numpy as np
import pandas as pd
import torch
from datasets import Dataset, DatasetDict
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)

from src.preprocessing import infer_label_from_score


In [ ]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Training device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


## Load dan Validasi Dataset


In [ ]:
LABEL_ALIASES = {
    "positif": "positive",
    "positive": "positive",
    "negatif": "negative",
    "negative": "negative",
    "netral": "neutral",
    "neutral": "neutral",
}

df = pd.read_csv(DATA_PATH)
df["content_clean"] = df["content_clean"].fillna("").astype(str)

if "label" not in df.columns:
    df["label"] = df["score"].apply(infer_label_from_score)
else:
    df["label"] = df["label"].fillna("").astype(str).str.lower().map(LABEL_ALIASES)
    mask_missing = df["label"].isna() | (df["label"].astype(str).str.strip() == "")
    df.loc[mask_missing, "label"] = df.loc[mask_missing, "score"].apply(infer_label_from_score)

df = df[df["content_clean"].str.strip() != ""].copy()
df = df[df["label"].isin(["negative", "neutral", "positive"])].copy()
df = df.drop_duplicates(subset=["content_clean", "label"]).reset_index(drop=True)

df["label"].value_counts()


Ubah `USE_NEUTRAL` menjadi `False` jika Anda ingin skema klasifikasi biner.


In [ ]:
USE_NEUTRAL = True

if not USE_NEUTRAL:
    df = df[df["label"] != "neutral"].copy()

label_list = sorted(df["label"].unique())
label2id = {label: index for index, label in enumerate(label_list)}
id2label = {index: label for label, index in label2id.items()}

df["labels"] = df["label"].map(label2id)
df[["content_clean", "label", "labels"]].head()


## Split Dataset


In [ ]:
if df["label"].value_counts().min() < 2:
    raise ValueError("Setiap kelas minimal harus memiliki 2 sampel untuk proses split.")

train_df, eval_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["label"],
)

dataset = DatasetDict(
    {
        "train": Dataset.from_pandas(train_df[["content_clean", "labels"]].reset_index(drop=True), preserve_index=False),
        "eval": Dataset.from_pandas(eval_df[["content_clean", "labels"]].reset_index(drop=True), preserve_index=False),
    }
)

dataset


## Tokenisasi


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_batch(batch):
    return tokenizer(batch["content_clean"], truncation=True, max_length=256)

tokenized_dataset = dataset.map(tokenize_batch, batched=True)
tokenized_dataset = tokenized_dataset.remove_columns(["content_clean"])
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
tokenized_dataset


## Training Configuration


In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id,
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, predictions),
        "f1_weighted": f1_score(labels, predictions, average="weighted"),
    }

training_args = TrainingArguments(
    output_dir=str(MODEL_DIR / "checkpoints"),
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1_weighted",
    greater_is_better=True,
    logging_steps=25,
    save_total_limit=2,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["eval"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)


## Fine-tuning dan Simpan Model


In [ ]:
trainer.train()
evaluation_result = trainer.evaluate()
evaluation_result


In [ ]:
trainer.save_model(str(MODEL_DIR))
tokenizer.save_pretrained(str(MODEL_DIR))
print(f"Model saved to: {MODEL_DIR}")


## Inference Sederhana


In [ ]:
sample_texts = [
    "aplikasinya aman dan sangat membantu transaksi harian",
    "sering error saat login dan otp tidak masuk",
]

prediction_inputs = tokenizer(sample_texts, truncation=True, padding=True, return_tensors="pt")
prediction_inputs = {key: value.to(model.device) for key, value in prediction_inputs.items()}
model.eval()
with torch.no_grad():
    prediction_output = model(**prediction_inputs)
predicted_ids = prediction_output.logits.argmax(dim=-1).tolist()
predicted_labels = [id2label[predicted_id] for predicted_id in predicted_ids]

list(zip(sample_texts, predicted_labels))


## Ekspor Hasil Prediksi ke `data/final`


In [ ]:
from src.sentiment import export_sentiment_results, predict_sentiments, prepare_prediction_rows

prediction_rows = prepare_prediction_rows()
prediction_results = predict_sentiments(prediction_rows, batch_size=16)
export_sentiment_results(prediction_results)
prediction_results[:3]
